# Data Sources and Inputs

The workflow depends on several source families: environmental products, fishing activity, telemetry records, and reference layers. This section explains what each source contributes, where the active configuration lives, and which reusable derived products are archived separately from the Git repository.


## Input Families

- Environmental data: SST, chlorophyll-a, SSH, wind, and bathymetry.
- Fisheries data: Global Fishing Watch AIS-derived apparent fishing activity.
- Biological data: SAERI telemetry records for BBAL and SAFS.
- Reference data: fisheries grid, FICZ/FOCZ, Natural Earth land/coastline.

In the Falkland case study, environmental and fisheries products cover 2014-2023, while telemetry records used here were collected during 2022-2023.


In [1]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
for name, spec in config.get("datasets", {}).items():
    role = spec.get("role", "unspecified")
    provider = spec.get("provider", "manual/local")
    print(f"{name:12s} role={role:16s} provider={provider}")


sst          role=environmental    provider=podaac
chl          role=environmental    provider=copernicus
ssh          role=environmental    provider=copernicus
wind         role=environmental    provider=cds
gfw          role=fishing_effort   provider=gfw
bathymetry   role=static           provider=gebco


## Repository, Raw Data, and Reference Layers

The Git repository tracks code, documentation, notebooks, lightweight configuration, and scripts. It does not track raw provider downloads, generated modeling products, generated plots, or downloaded reference layers.

Public reference layers are restored from their providers with:

```bash
python3 scripts/download_reference_data.py
```

Raw provider data remain local because they may require credentials, provider access approval, or redistribution checks. This includes raw environmental provider downloads, Global Fishing Watch source products, and source species-observation data.


## Zenodo Data Bundle

Reusable derived products are archived separately from the Git repository in the Zenodo data bundle:

<https://doi.org/10.5281/zenodo.20334807>

The bundle is intended for products that are useful to inspect, reuse, or run examples without rebuilding the entire workflow from raw provider data. The current bundle focuses on the H3 study grid, processed lookup/index tables, selected production predictions, weekly operational products, the consolidated environmental-regime table, plausibility products, curated metrics, selected plot exports, and generated figures.

The consolidated environmental-regime table is stored under `modeling/environmental_regimes/`. It contains selected SOM hierarchical k30 seascape assignments (`som_prototype`, `seascape`, `seascape_distance`) and Bayesian GMM k30 component assignments (`bayesian_gmm_k30_component`, probability, and entropy).


## Products Not Redistributed

The Zenodo bundle is not a mirror of every local file. It is intended to provide the reusable derived inputs needed to inspect and rerun the public workflow without redistributing source material that should be obtained from its original provider.

Not included in the data bundle:

- raw provider downloads under `data/raw/`
- local credentials or authentication files
- reference layers that can be restored with `scripts/download_reference_data.py`
- intermediate diagnostics and exploratory products that are regenerated by the build or plotting scripts
- source datasets with sensitive, restricted, or provider-dependent redistribution terms

The retained `modeling/seascape_species_use/` product is limited to the 2022 surface used by the public seascape-conditioned matrices. It can be regenerated with:

```bash
python3 scripts/build/build_seascape_species_use_surfaces.py --years 2022
```


## Credentials

Provider credentials must remain outside committed files. See `docs/authentication.md` for the current convention: Earthdata in `~/.netrc`, Copernicus Marine through `copernicusmarine login`, CDS in `~/.cdsapirc`, and Global Fishing Watch in local `.env` as `GFW_TOKEN`.
